In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [2]:
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset, random_split, TensorDataset
import torch.nn.functional as torch_F # avoid import problems becasue F is used as a variable

import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score

import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

import sys
sys.path.append('../src')

from preprocessing import *
from models import  *
from utils import *

from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

/home/waasiq/miniconda3/envs/alpha/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device(type='cuda')

In [3]:
dfs = get_dfs(os.path.dirname(os.getcwd()))
static_df = create_static_df(dfs)
medication_df = create_medication_df(dfs)
vitals_ca, vitals_lab = create_vitals_df(dfs)
biopsy_df = dfs['biopsy']

ts_data = create_ts_data(vitals_ca, vitals_lab, medication_df, merge_lab=True, merge_med=True, static_df=static_df)

# notes = create_notes_df(dfs, filename='../data/embeddings/emb_gte.npy')
notes = create_notes_df(dfs, filename='../data/embeddings/emb_med_gte_simcse_en_ger.npy')
# notes = create_notes_df(dfs, filename=None)

pool_assignments_path = os.path.join('..', 'data', 'splits', 'pool_assignments.json')
with open(pool_assignments_path, 'r', encoding='utf-8') as f:
    pool_assignments = json.load(f)

pool_a_patient_ids = np.asarray(pool_assignments['pool_a'])
all_valid_patient_ids = get_valid_patient_ids(
    static_df=static_df,
    ts_data=ts_data,
    notes_df=notes,
    min_ts_count=10,
    require_notes=True,
)
selected_pool_a_ids = np.asarray([pid for pid in all_valid_patient_ids if pid in set(pool_a_patient_ids)])
pool_a_split_ids = split_patient_ids(
    patient_ids=selected_pool_a_ids,
    train_size=0.9,
    val_size=0.1,
    test_size=0.0,
    random_state=42,
    shuffle=True,
)

train_dataset = NephroCAGEDataset(
    static_df=static_df,
    ts_data=ts_data,
    notes_df=notes,
    biopsy_df=biopsy_df,
    patient_ids=pool_a_split_ids['train'],
    fit_preprocessing=True,
    min_ts_count=10,
    require_notes=True,
)
preprocessing_artifacts = train_dataset.preprocessing_artifacts

val_dataset = NephroCAGEDataset(
    static_df=static_df,
    ts_data=ts_data,
    notes_df=notes,
    biopsy_df=biopsy_df,
    patient_ids=pool_a_split_ids['val'],
    preprocessing_artifacts=preprocessing_artifacts,
    fit_preprocessing=False,
    min_ts_count=10,
    require_notes=True,
)

ts_scaler = train_dataset.ts_scaler
static_scaler = train_dataset.scaler

train_ids = set(pool_a_split_ids['train'])
val_ids = set(pool_a_split_ids['val'])
selected_ids = set(selected_pool_a_ids.tolist())

assert train_ids.isdisjoint(val_ids), 'Data leakage: train overlaps val'
assert (train_ids | val_ids) == selected_ids, 'Split IDs do not cover selected cohort exactly'

print(f"Pool A size: {len(pool_a_patient_ids)}")
print(f"Selected after eligibility filtering: {len(selected_pool_a_ids)}")
print(f"Train/Val sizes: {len(train_ids)} / {len(val_ids)}")
print('Validation split is used only for model checkpointing.')


Unique patients in medication: 3335
Removing patients that are not in static_df
Unique patients in clinical assessments: 3296
Average entries per patient 78.9
Unique patients in clinical assessments: 3465
Removing patients that are not in static_df
Unique patients in clinical assessments: 3423
Average entries per patient 61.16155419222904
Unique patients in lab df: 3460
Removing patients that are not in static_df
Unique patients in lab df: 3410
Average entries per patient 472.2140762463343


KeyboardInterrupt: 

In [ ]:
batch_size = 16
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

# vanilla_lstm = VanillaTimeSeriesEncoder()
att_encoder = TimeAwareAttentionEncoder(use_temporal_attention=True)
model = MultiModal(att_encoder, categorical_cardinalities=train_dataset.categorical_cardinalities, use_static=True, use_notes=True).to(device)
predict_steps_ahead = 1
lambda_corr = 0.0
lambda_modality_mi = 0.0
lambda_feature_loss = 0.0


In [5]:
criterion = nn.MSELoss(reduction='none')
optimizer = optim.Adam(model.parameters(), lr=0.0003)

feature_decoders = FeatureDecoders(CONFIG['lstm_hidden_size'], CONFIG).to(device)
feature_decoder_optimizer = optim.Adam(feature_decoders.parameters(), lr=0.0003)

num_epochs = 30
min_delta = 1e-4
best_val_loss = float('inf')
best_epoch = -1
save_path = '../models/full10ep_poola_701020_best.pt'

for epoch in range(num_epochs):
    epoch_loss = 0.0
    total_valid_points = 0.0
    model.train()

    with tqdm(total=len(train_dataloader), desc=f'Train Epoch {epoch+1}/{num_epochs}', leave=False) as pbar:
        for batch in train_dataloader:
            cat_features = batch['static_categorical_features'].to(device)
            num_features_input = batch['static_numerical_features'].to(device)

            full_seq = batch['ts_features'].to(device)
            timesteps = batch['timesteps'].to(device)
            value_mask_full = batch['value_mask'].to(device)
            mask = batch['mask'].to(device)
            B, T, F = full_seq.shape

            notes_embeddings = batch['notes_embeddings'].to(device)
            notes_timesteps = batch['notes_timesteps'].to(device)
            notes_mask = batch['notes_mask'].to(device)

            if T <= predict_steps_ahead:
                pbar.update(1)
                continue

            input_seq = full_seq[:, :-predict_steps_ahead, :]
            input_timesteps = timesteps[:, :-predict_steps_ahead]
            delta_times = timesteps[:, 1:] - timesteps[:, :-1]
            elapsed_times = delta_times[:, :(T - predict_steps_ahead)]

            input_mask = mask[:, :-predict_steps_ahead]
            input_value_mask = value_mask_full[:, :-predict_steps_ahead, :]

            optimizer.zero_grad()
            feature_decoder_optimizer.zero_grad()

            outputs, hidden_states, _, static_embedding = model(
                x=input_seq,
                elapsed_times=elapsed_times,
                timesteps=input_timesteps,
                notes_embeddings=notes_embeddings,
                notes_timesteps=notes_timesteps,
                static_features=(cat_features, num_features_input),
                mask=input_mask,
                value_mask=input_value_mask,
                notes_mask=notes_mask
            )

            target_seq = full_seq[:, predict_steps_ahead:, :]
            valid_mask = value_mask_full[:, predict_steps_ahead:, :]

            raw_loss = criterion(outputs, target_seq)
            masked_loss = raw_loss * valid_mask.float()
            valid_points = valid_mask.float().sum()

            accum_loss = masked_loss.sum()
            final_loss = accum_loss / valid_points if valid_points > 0 else torch.tensor(0.0, device=device)

            last_hidden = get_last_valid_step(hidden_states, input_mask)
            last_notes = get_last_valid_note_embedding(notes_embeddings, notes_mask)
            last_ts = get_last_valid_step(full_seq, mask)
            decorr_loss = correlation_loss(last_hidden)

            mi_loss = modality_mi_loss(
                last_hidden,
                static_embedding,
                last_notes
            )

            feature_preds, masks = feature_decoders(last_hidden)
            feat_pred_loss = feature_prediction_loss(
                feature_preds,
                masks,
                cat_features,
                num_features_input,
                last_ts
            )

            batch_loss = final_loss + lambda_corr * decorr_loss + lambda_modality_mi * mi_loss  # + lambda_feature_loss * feat_pred_loss
            batch_loss.backward()
            optimizer.step()
            feature_decoder_optimizer.step()

            epoch_loss += final_loss.item() * valid_points.item()
            total_valid_points += valid_points.item()

            avg_loss_so_far = epoch_loss / total_valid_points if total_valid_points > 0 else 0.0
            pbar.set_postfix({
                'avg_train_loss': f'{avg_loss_so_far:.4f}',
                'corr': f'{decorr_loss.item():.4f}',
                'mi': f'{mi_loss.item():.4f}'
            })
            pbar.update(1)

    avg_train_loss = epoch_loss / total_valid_points if total_valid_points > 0 else float('inf')

    model.eval()
    val_loss_sum = 0.0
    val_valid_points = 0.0

    with torch.no_grad():
        for batch in val_dataloader:
            cat_features = batch['static_categorical_features'].to(device)
            num_features_input = batch['static_numerical_features'].to(device)

            full_seq = batch['ts_features'].to(device)
            timesteps = batch['timesteps'].to(device)
            value_mask_full = batch['value_mask'].to(device)
            mask = batch['mask'].to(device)
            B, T, F = full_seq.shape

            notes_embeddings = batch['notes_embeddings'].to(device)
            notes_timesteps = batch['notes_timesteps'].to(device)
            notes_mask = batch['notes_mask'].to(device)

            if T <= predict_steps_ahead:
                continue

            input_seq = full_seq[:, :-predict_steps_ahead, :]
            input_timesteps = timesteps[:, :-predict_steps_ahead]
            delta_times = timesteps[:, 1:] - timesteps[:, :-1]
            elapsed_times = delta_times[:, :(T - predict_steps_ahead)]

            input_mask = mask[:, :-predict_steps_ahead]
            input_value_mask = value_mask_full[:, :-predict_steps_ahead, :]

            outputs, _, _, _ = model(
                x=input_seq,
                elapsed_times=elapsed_times,
                timesteps=input_timesteps,
                notes_embeddings=notes_embeddings,
                notes_timesteps=notes_timesteps,
                static_features=(cat_features, num_features_input),
                mask=input_mask,
                value_mask=input_value_mask,
                notes_mask=notes_mask
            )

            target_seq = full_seq[:, predict_steps_ahead:, :]
            valid_mask = value_mask_full[:, predict_steps_ahead:, :]

            raw_loss = criterion(outputs, target_seq)
            masked_loss = raw_loss * valid_mask.float()
            valid_points = valid_mask.float().sum()

            val_loss_sum += masked_loss.sum().item()
            val_valid_points += valid_points.item()

    avg_val_loss = val_loss_sum / val_valid_points if val_valid_points > 0 else float('inf')
    improved = avg_val_loss < (best_val_loss - min_delta)

    if improved:
        best_val_loss = avg_val_loss
        best_epoch = epoch + 1
        torch.save(model.state_dict(), save_path)
        ckpt_flag = ' *best*'
    else:
        ckpt_flag = ''

    print(
        f'Epoch {epoch+1}/{num_epochs} | train_loss={avg_train_loss:.4f} | '
        f'val_loss={avg_val_loss:.4f} | best_val={best_val_loss:.4f} | '
        f'best_epoch={best_epoch}{ckpt_flag}'
    )

model.load_state_dict(torch.load(save_path, weights_only=True))
print(
    f'Training completed. '
    f'Loaded best checkpoint from epoch {best_epoch}: {save_path}'
)


Epoch 1/30 | train_loss=0.5574 | val_loss=0.4390 | best_val=0.4390 | best_epoch=1 *best*


Epoch 2/30 | train_loss=0.4044 | val_loss=0.3964 | best_val=0.3964 | best_epoch=2 *best*


Epoch 3/30 | train_loss=0.3800 | val_loss=0.3774 | best_val=0.3774 | best_epoch=3 *best*


Epoch 4/30 | train_loss=0.3591 | val_loss=0.3657 | best_val=0.3657 | best_epoch=4 *best*


Epoch 5/30 | train_loss=0.3460 | val_loss=0.3554 | best_val=0.3554 | best_epoch=5 *best*


...

Epoch 26/30 | train_loss=0.2755 | val_loss=0.3420 | best_val=0.3297 | best_epoch=17


Epoch 27/30 | train_loss=0.2740 | val_loss=0.3411 | best_val=0.3297 | best_epoch=17


Epoch 28/30 | train_loss=0.2713 | val_loss=0.3424 | best_val=0.3297 | best_epoch=17


Epoch 29/30 | train_loss=0.2659 | val_loss=0.3497 | best_val=0.3297 | best_epoch=17


Epoch 30/30 | train_loss=0.2619 | val_loss=0.3461 | best_val=0.3297 | best_epoch=17
Training completed. Loaded best checkpoint from epoch 17: ../models/full10ep_poola_701020_best.pt


In [ ]:
model.eval()
with torch.no_grad():
    for batch in train_dataloader:
        cat_features = batch['static_categorical_features'].to(device)
        num_features = batch['static_numerical_features'].to(device)

        full_seq = batch['ts_features'].to(device)       # (B, T, F)
        timesteps = batch['timesteps'].to(device)        # (B, T)
        value_mask_full = batch['value_mask'].to(device) # (B, T, F)

        notes_embeddings = batch['notes_embeddings'].to(device)
        notes_timesteps = batch['notes_timesteps'].to(device)
        notes_mask = batch['notes_mask'].to(device)

        B, T, F = full_seq.shape

        # Skip if sequence is too short for the desired horizon
        if T <= predict_steps_ahead:
            continue

        # Input => (B, T - n, F)
        input_seq = full_seq[:, :-predict_steps_ahead, :]
        # Target => (B, T - n, F)
        target_seq = full_seq[:, predict_steps_ahead:, :]

        # Elapsed times => (B, T-1), keep only first (T - n)
        delta_times = timesteps[:, 1:] - timesteps[:, :-1]  # shape (B, T-1)
        elapsed_times = delta_times[:, : (T - predict_steps_ahead)]

        input_value_mask = value_mask_full[:, :-predict_steps_ahead, :]

        # Value mask => for the same target portion
        value_mask = value_mask_full[:, predict_steps_ahead:, :]

        outputs, _, _, _ = model(
            x=input_seq,
            elapsed_times=elapsed_times,
            timesteps=timesteps[:, :-predict_steps_ahead],
            notes_embeddings=notes_embeddings,
            notes_timesteps=notes_timesteps,
            static_features=(cat_features, num_features),
            mask=batch['mask'][:, :-predict_steps_ahead].to(device),  # (B, T - n)
            value_mask=input_value_mask,
            notes_mask=notes_mask
        )
        # outputs => (B, T - n, F), normalized

        # Bring to CPU for plotting / inverse transform
        outputs_np = outputs.cpu().numpy()
        target_np = target_seq.cpu().numpy()
        value_mask_np = value_mask.cpu().numpy()
        timesteps_np = timesteps.cpu().numpy()

        B_, TmN, F_ = outputs_np.shape  # TmN = T - n

        # Reshape for scaler
        outputs_2d = outputs_np.reshape(-1, F_)
        target_2d = target_np.reshape(-1, F_)

        # Apply inverse scaling
        outputs_orig = ts_scaler.inverse_transform(outputs_2d).reshape(B_, TmN, F_)
        target_orig = ts_scaler.inverse_transform(target_2d).reshape(B_, TmN, F_)

        patient_idx = 2
        pred_values = outputs_orig[patient_idx]    # shape (T - n, F)
        true_values = target_orig[patient_idx]     # shape (T - n, F)
        mask_vals = value_mask_np[patient_idx]     # shape (T - n, F)

        # Time steps for those predicted points: (T - n) steps from index n onward
        patient_timesteps = timesteps_np[patient_idx, predict_steps_ahead:]  # shape (T - n,)

        feature_idx = list(range(len(CONFIG['ts_features'])))
        num_features_to_plot = len(feature_idx)
        fig, axs = plt.subplots(num_features_to_plot, 1, figsize=(10, 6 * num_features_to_plot))

        # Handle single axis vs multiple
        if num_features_to_plot == 1:
            axs = [axs]

        for i, f_idx in enumerate(feature_idx):
            valid_indices = (mask_vals[:, f_idx] == 1)

            axs[i].scatter(patient_timesteps[valid_indices],
                           true_values[valid_indices, f_idx],
                           color='blue', label='Actual', alpha=0.7)
            axs[i].plot(patient_timesteps[valid_indices],
                        true_values[valid_indices, f_idx],
                        color='blue', alpha=0.3, linestyle='--')

            axs[i].scatter(patient_timesteps[valid_indices],
                           pred_values[valid_indices, f_idx],
                           color='red', label='Predicted', alpha=0.7)
            axs[i].plot(patient_timesteps[valid_indices],
                        pred_values[valid_indices, f_idx],
                        color='red', alpha=0.3, linestyle='--')

            feature_name = CONFIG['ts_features'][f_idx]
            axs[i].set_title(f"{feature_name} Predictions vs Actual (N-step={predict_steps_ahead})")
            axs[i].set_xlabel("Relative Time (days)")
            axs[i].set_ylabel(feature_name)
            axs[i].legend()

        plt.tight_layout()
        plt.show()

        # Just visualize for the first batch and stop
        break
